# HGTCM P5 - Full Run Initiator

This notebook remains the readable orchestrator for the canonical target workflow. P3 correction now supports both notebook execution and the canonical script path so the shared codebase stays consistent with the matched BART reruns, while the An Gaodhal archive stays outside the default run path.

## Imports And Run Settings


In [ ]:
# ## Initiator env: which stages to run, run labels, workspace roots for P2 and P3 outputs.
# Why: fixed stage flags make runs reproducible and easier to compare across hardware/time windows.
import hashlib
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path


def env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


def file_sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_jsonl_head(path: Path, limit: int = 20) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
            if len(rows) >= limit:
                break
    return rows


ROOT = Path(os.environ.get("PROJECT_DECODE_ROOT", ".")).resolve()
NOTEBOOK_ROOT = Path(os.environ.get("PROJECT_DECODE_NOTEBOOK_ROOT", ROOT / "scratch_target" / "notebooks")).resolve()
SCRIPT_ROOT = Path(os.environ.get("PROJECT_DECODE_SCRIPT_ROOT", ROOT / "scratch_target" / "scripts")).resolve()
LAUNCHER_ROOT = Path(os.environ.get("PROJECT_DECODE_LAUNCHER_ROOT", ROOT / "scratch_target" / "launchers")).resolve()
os.environ["PROJECT_DECODE_ROOT"] = str(ROOT)
os.environ.setdefault("PROJECT_DECODE_NOTEBOOK_ROOT", str(NOTEBOOK_ROOT))
os.environ.setdefault("PROJECT_DECODE_SCRIPT_ROOT", str(SCRIPT_ROOT))
DATA_RUN_LABEL = os.environ.get("DATA_RUN_LABEL", "affine_edge_full_clamped").strip() or "affine_edge_full_clamped"
os.environ["DATA_RUN_LABEL"] = DATA_RUN_LABEL
if DATA_RUN_LABEL == "full":
    os.environ.setdefault("TARGET_ACCEPTED_PAGES", "784")
    os.environ.setdefault("MAX_SCAN_PAGES", "900")
    os.environ.setdefault("PROJECTION_CLEANED_PAGE_RUN_LABEL", "projection_v1")
PIPELINE_DRY_RUN = env_flag("PIPELINE_DRY_RUN", "0")
RUN_P1 = env_flag("RUN_P1", "1" if DATA_RUN_LABEL == "full" else "0")
ALLOW_MISSING_P1 = env_flag("ALLOW_MISSING_P1", "0")
ALLOW_UNREVIEWED_CROPS = env_flag("ALLOW_UNREVIEWED_CROPS", "0")
NOTEBOOK_TIMEOUT_SECONDS = int(os.environ.get("PIPELINE_NOTEBOOK_TIMEOUT", "72000"))
RUN_ID = os.environ.get("PIPELINE_RUN_ID") or datetime.now(timezone.utc).strftime("full_run_%Y%m%dT%H%M%SZ")
RUN_DIR = ROOT / "runs" / "full_pipeline" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

P3_EXECUTION_MODE = os.environ.get("P3_EXECUTION_MODE", "notebook").strip().lower() or "notebook"
P3_SCRIPT_NAME = os.environ.get("P3_SCRIPT_NAME", "run_target_only_bart_control_server.sh").strip() or "run_target_only_bart_control_server.sh"

os.environ.setdefault("RUN_ANGAODHAL_REFERENCE_TRAINING", "0")
os.environ.setdefault("RUN_TRANSFER_TRAINING", "0")
os.environ.setdefault("RUN_FROZEN_BART", "1")
os.environ.setdefault("RUN_BART_FINETUNE", "1")

ACTIVE_NOTEBOOKS = {
    "P1": ["P1_Target_Preprocess_and_Assign.ipynb"],
    "P2": ["P2_Target_OCR_Train_and_Predict.ipynb"],
    "P3": ["P3_Target_BART_Correction.ipynb"],
    "P4": ["P4_Target_Score_Corrections.ipynb"],
}

print(json.dumps({
    "root": str(ROOT),
    "data_run_label": DATA_RUN_LABEL,
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "dry_run": PIPELINE_DRY_RUN,
    "run_p1": RUN_P1,
    "timeout_seconds": NOTEBOOK_TIMEOUT_SECONDS,
    "p3_execution_mode": P3_EXECUTION_MODE,
    "p3_script_name": P3_SCRIPT_NAME,
}, indent=2))


## Preflight Checks


In [ ]:
# ## subprocess helpers: run notebook via jupyter, stream logs, optional cloud sync hooks.
def resolve_notebook(stage_id: str) -> Path:
    override = os.environ.get(f"{stage_id}_NOTEBOOK")
    candidates = [override] if override else ACTIVE_NOTEBOOKS[stage_id]
    for name in candidates:
        if not name:
            continue
        path = (NOTEBOOK_ROOT / name).resolve()
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not resolve {stage_id} notebook from {candidates}")


stage_notebooks = {stage_id: resolve_notebook(stage_id) for stage_id in ACTIVE_NOTEBOOKS}
target_manifest = ROOT / "manifests" / f"target_lines_{DATA_RUN_LABEL}.jsonl"
target_crop_dir = ROOT / "data" / "derived" / "target_line_crops" / DATA_RUN_LABEL
projection_cleaned_dir = ROOT / "data" / "derived" / "target_cleaned_pages" / "projection_v1"
target_contact_sheet = ROOT / "data" / "derived" / f"target_contact_sheet_{DATA_RUN_LABEL}.png"
requirements_file = ROOT / "requirements-trocr.txt"

required_paths = {
    "target_manifest": target_manifest,
    "target_crop_dir": target_crop_dir,
    "projection_cleaned_pages": projection_cleaned_dir,
    "requirements_file": requirements_file,
}

status = {}
for key, path in required_paths.items():
    if path.is_dir():
        png_count = len(list(path.glob("*.png")))
        status[key] = {"path": str(path), "exists": True, "png_count": png_count}
    else:
        status[key] = {"path": str(path), "exists": path.exists(), "size": path.stat().st_size if path.exists() else 0}

status["target_contact_sheet"] = {
    "path": str(target_contact_sheet),
    "exists": target_contact_sheet.exists(),
    "required": not ALLOW_UNREVIEWED_CROPS,
}

WIN_PREFIX = "C:\\Users\\Jobias\\OneDrive\\Desktop\\Project Decode\\"


def portable_path(value: str | Path) -> Path:
    raw = str(value)
    if raw.startswith(WIN_PREFIX):
        return ROOT.joinpath(*raw[len(WIN_PREFIX):].split("\\"))
    path = Path(raw)
    if path.exists():
        return path
    if not path.is_absolute():
        return (ROOT / path).resolve()
    return path


path_probe_rows = read_jsonl_head(target_manifest, 20)
missing_images = []
for row in path_probe_rows:
    image_path = portable_path(row.get("image_path", ""))
    if not image_path.exists():
        missing_images.append({"sample_id": row.get("sample_id"), "image_path": str(image_path)})
status["image_path_probe"] = {"checked": len(path_probe_rows), "missing": missing_images[:10]}

try:
    import torch
    cuda_available = bool(torch.cuda.is_available())
    gpu_count = int(torch.cuda.device_count()) if cuda_available else 0
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)] if cuda_available else []
except Exception as exc:
    cuda_available = False
    gpu_count = 0
    gpu_names = []
    status["torch_probe_error"] = repr(exc)

status["compute"] = {"cuda_available": cuda_available, "gpu_count": gpu_count, "gpu_names": gpu_names}
status["wandb"] = {
    "USE_WANDB": os.environ.get("USE_WANDB"),
    "WANDB_MODE": os.environ.get("WANDB_MODE"),
    "WANDB_PROJECT": os.environ.get("WANDB_PROJECT"),
}
status["huggingface"] = {
    "HF_TOKEN_present": bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")),
}
status["notebooks"] = {stage: str(path) for stage, path in stage_notebooks.items()}
status["scripts"] = {
    'build_correction_pairs_from_predictions': str(SCRIPT_ROOT / 'build_correction_pairs_from_predictions.py'),
    'export_target_split_predictions': str(SCRIPT_ROOT / 'export_target_split_predictions.py'),
    'run_bart_correction_experiment': str(SCRIPT_ROOT / 'run_bart_correction_experiment.py'),
    'run_bart_correction_server': str(SCRIPT_ROOT / 'run_bart_correction_server.sh'),
    'run_target_only_bart_control_server': str(SCRIPT_ROOT / 'run_target_only_bart_control_server.sh'),
    'run_tesseract_bart_control_server': str(SCRIPT_ROOT / 'run_tesseract_bart_control_server.sh'),
}
status["launchers"] = {
    'launch_target_ocr_clean': str(LAUNCHER_ROOT / 'launch_target_ocr_clean.sh'),
    'launch_target_refresh_bundle': str(LAUNCHER_ROOT / 'launch_target_refresh_bundle.sh'),
    'launch_scratch_target_full': str(LAUNCHER_ROOT / 'launch_scratch_target_full.sh'),
}

print(json.dumps(status, indent=2))

missing = [key for key, item in status.items() if isinstance(item, dict) and item.get("exists") is False and key != "target_contact_sheet"]
empty_dirs = [key for key, item in status.items() if isinstance(item, dict) and "png_count" in item and item["png_count"] == 0]
if status["target_contact_sheet"]["required"] and not status["target_contact_sheet"]["exists"]:
    missing.append("target_contact_sheet")
if missing_images:
    missing.append("manifest_image_paths")

if RUN_P1:
    p1_generated_keys = {"target_manifest", "target_crop_dir", "projection_cleaned_pages", "target_contact_sheet", "manifest_image_paths"}
    missing = [key for key in missing if key not in p1_generated_keys]
    empty_dirs = []

if (missing or empty_dirs) and not ALLOW_MISSING_P1:
    raise SystemExit(
        "P1 outputs are not ready or portable paths failed. "
        f"Set ALLOW_MISSING_P1=1 only for dry setup. Missing={missing}, empty_dirs={empty_dirs}"
    )


## Source Provenance Manifest


In [ ]:
# ## Stage ordering and artifact paths: where each notebook handoff is written.
provenance = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "root": str(ROOT),
    "data_run_label": DATA_RUN_LABEL,
    "study_scope": {
        "canonical_target_workflow": True,
        "include_angaodhal_archive_reference": False,
    },
    "active_notebooks": {
        stage: {
            "path": str(path),
            "size": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for stage, path in stage_notebooks.items()
    },
    "inputs": {
        "target_manifest": str(target_manifest),
        "target_crop_dir": str(target_crop_dir),
        "projection_cleaned_pages": str(projection_cleaned_dir),
        "target_contact_sheet": str(target_contact_sheet),
        "requirements_file": str(requirements_file),
    },
    "hf_models": {
        "ocr_default": os.environ.get("OCR_MODEL_NAME", "microsoft/trocr-base-printed"),
        "bart_correction_preferred": "ancatmara/bart-ocr-correction-ga-en",
        "bart_correction_fallback": "facebook/bart-base",
        "irish_bert_note": "Encoder/fill-mask Irish BERT models are optional reranking/scoring sources, not direct correction generators.",
    },
    "script_controls": {
        "p3_execution_mode": P3_EXECUTION_MODE,
        "p3_script_name": P3_SCRIPT_NAME,
        "run_bart_correction_experiment": str(SCRIPT_ROOT / 'run_bart_correction_experiment.py'),
        "run_bart_correction_server": str(SCRIPT_ROOT / 'run_bart_correction_server.sh'),
        "run_target_only_bart_control_server": str(SCRIPT_ROOT / 'run_target_only_bart_control_server.sh'),
        "run_tesseract_bart_control_server": str(SCRIPT_ROOT / 'run_tesseract_bart_control_server.sh'),
    },
    "output_contracts": {
        "canonical_ocr_predictions": str(ROOT / "runs" / "ocr_training" / "latest" / "target_predictions.jsonl"),
        "frozen_bart_predictions": "reports/correction_bart_<run_id>/corrected_predictions_frozen_bart.jsonl",
        "finetuned_bart_predictions": "reports/correction_bart_<run_id>/corrected_predictions_finetuned_bart.jsonl",
        "final_scorecard": str(ROOT / "reports" / f"{DATA_RUN_LABEL}_final_correction_scorecard.csv"),
    },
    "preflight_status": status,
}
provenance_path = RUN_DIR / "source_provenance_manifest.json"
provenance_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2) + "
", encoding="utf-8")
print(f"Source provenance manifest written to {provenance_path}")


## Execute P2-P4 Or Canonical P3 Correction Scripts

In [ ]:


def resolve_script(script_name: str) -> Path:
    path = (SCRIPT_ROOT / script_name).resolve()
    if not path.exists():
        raise FileNotFoundError(f'Could not resolve script: {path}')
    return path


def execute_script_subprocess(stage_id: str, script_path: Path, *, env_overrides: dict[str, str] | None = None) -> dict:
    env = os.environ.copy()
    env.setdefault('PROJECT_DECODE_ROOT', str(ROOT))
    env.setdefault('DATA_RUN_LABEL', DATA_RUN_LABEL)
    if env_overrides:
        env.update({key: str(value) for key, value in env_overrides.items()})
    started = datetime.now(timezone.utc).isoformat()
    start = time.time()
    summary = {
        'stage_id': stage_id,
        'script_path': str(script_path),
        'env_overrides': env_overrides or {},
        'started_at': started,
        'status': 'pending',
    }
    if PIPELINE_DRY_RUN:
        summary['status'] = 'dry_run'
        summary['finished_at'] = datetime.now(timezone.utc).isoformat()
        summary['duration_seconds'] = 0
        return summary
    try:
        subprocess.run(['bash', str(script_path)], cwd=str(ROOT), env=env, check=True)
        summary['status'] = 'completed'
    except Exception as exc:
        summary['status'] = 'failed'
        summary['error'] = repr(exc)
        raise
    finally:
        summary['finished_at'] = datetime.now(timezone.utc).isoformat()
        summary['duration_seconds'] = round(time.time() - start, 3)
    return summary


# ## Main driver: execute configured pipeline steps in order and summarize exit codes / paths.
# Why: explicit stage ordering preserves causal provenance of artifacts used by downstream scoring.
def execute_notebook_subprocess(stage_id: str, notebook_path: Path, *, env_overrides: dict[str, str] | None = None) -> dict:
    output_path = RUN_DIR / f"{stage_id}_{notebook_path.stem}.executed.ipynb"
    env = os.environ.copy()
    env.setdefault("PROJECT_DECODE_ROOT", str(ROOT))
    env.setdefault("DATA_RUN_LABEL", DATA_RUN_LABEL)
    if env_overrides:
        env.update({key: str(value) for key, value in env_overrides.items()})
    started = datetime.now(timezone.utc).isoformat()
    start = time.time()
    summary = {
        "stage_id": stage_id,
        "notebook_path": str(notebook_path),
        "output_path": str(output_path),
        "env_overrides": env_overrides or {},
        "started_at": started,
        "status": "pending",
    }
    if PIPELINE_DRY_RUN:
        summary["status"] = "dry_run"
        summary["finished_at"] = datetime.now(timezone.utc).isoformat()
        summary["duration_seconds"] = 0
        return summary
    command = [
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(notebook_path),
        "--output",
        output_path.stem,
        "--output-dir",
        str(output_path.parent),
        f"--ExecutePreprocessor.timeout={NOTEBOOK_TIMEOUT_SECONDS}",
        f"--ExecutePreprocessor.kernel_name={os.environ.get('PIPELINE_KERNEL_NAME', 'python3')}",
    ]
    try:
        subprocess.run(command, cwd=str(ROOT), env=env, check=True)
        summary["status"] = "completed"
    except Exception as exc:
        summary["status"] = "failed"
        summary["error"] = repr(exc)
        raise
    finally:
        summary["finished_at"] = datetime.now(timezone.utc).isoformat()
        summary["duration_seconds"] = round(time.time() - start, 3)
    return summary


pipeline_summary = {
    "run_id": RUN_ID,
    "root": str(ROOT),
    "data_run_label": DATA_RUN_LABEL,
    "dry_run": PIPELINE_DRY_RUN,
    "started_at": datetime.now(timezone.utc).isoformat(),
    "stages": [],
}

run_p2 = env_flag("RUN_P2", "1")
run_p3 = env_flag("RUN_P3", "1")
run_p4 = env_flag("RUN_P4", "1")
parallel_correction = env_flag("PIPELINE_PARALLEL_CORRECTION", "0")

if RUN_P1:
    pipeline_summary["stages"].append(execute_notebook_subprocess("P1", stage_notebooks["P1"]))
    if not PIPELINE_DRY_RUN and not target_manifest.exists():
        raise SystemExit(f"P1 completed but did not write expected target manifest: {target_manifest}")
else:
    pipeline_summary["stages"].append({"stage_id": "P1", "status": "skipped"})

if run_p2:
    pipeline_summary["stages"].append(execute_notebook_subprocess("P2", stage_notebooks["P2"]))
else:
    pipeline_summary["stages"].append({"stage_id": "P2", "status": "skipped"})

if run_p3:
    if P3_EXECUTION_MODE == 'script':
        script_path = resolve_script(P3_SCRIPT_NAME)
        pipeline_summary['stages'].append(execute_script_subprocess('P3_script', script_path))
    elif parallel_correction:
        if gpu_count < 2 and not os.environ.get("P3_FINETUNED_GPU"):
            raise SystemExit("PIPELINE_PARALLEL_CORRECTION=1 requires at least 2 GPUs or explicit P3_*_GPU assignments.")
        jobs = [
            ("P3_frozen", {"RUN_FROZEN_BART": "1", "RUN_BART_FINETUNE": "0", "CUDA_VISIBLE_DEVICES": os.environ.get("P3_FROZEN_GPU", "0")}),
            ("P3_finetuned", {"RUN_FROZEN_BART": "0", "RUN_BART_FINETUNE": "1", "CUDA_VISIBLE_DEVICES": os.environ.get("P3_FINETUNED_GPU", "1")}),
        ]
        processes = []
        for stage_id, overrides in jobs:
            output_path = RUN_DIR / f"{stage_id}_{stage_notebooks['P3'].stem}.executed.ipynb"
            env = os.environ.copy()
            env.update({"PROJECT_DECODE_ROOT": str(ROOT), "DATA_RUN_LABEL": DATA_RUN_LABEL, **overrides})
            command = [
                "jupyter", "nbconvert", "--to", "notebook", "--execute", str(stage_notebooks["P3"]),
                "--output", output_path.stem, "--output-dir", str(output_path.parent),
                f"--ExecutePreprocessor.timeout={NOTEBOOK_TIMEOUT_SECONDS}",
                f"--ExecutePreprocessor.kernel_name={os.environ.get('PIPELINE_KERNEL_NAME', 'python3')}",
            ]
            summary = {"stage_id": stage_id, "notebook_path": str(stage_notebooks["P3"]), "output_path": str(output_path), "env_overrides": overrides}
            if PIPELINE_DRY_RUN:
                summary["status"] = "dry_run"
                pipeline_summary["stages"].append(summary)
            else:
                processes.append((summary, subprocess.Popen(command, cwd=str(ROOT), env=env)))
        for summary, process in processes:
            start = time.time()
            return_code = process.wait()
            summary["duration_seconds"] = round(time.time() - start, 3)
            summary["status"] = "completed" if return_code == 0 else "failed"
            summary["return_code"] = return_code
            pipeline_summary["stages"].append(summary)
            if return_code != 0:
                raise SystemExit(f"{summary['stage_id']} failed with return code {return_code}")
    else:
        pipeline_summary["stages"].append(execute_notebook_subprocess("P3", stage_notebooks["P3"]))
else:
    pipeline_summary["stages"].append({"stage_id": "P3", "status": "skipped"})

if run_p4:
    pipeline_summary["stages"].append(execute_notebook_subprocess("P4", stage_notebooks["P4"]))
else:
    pipeline_summary["stages"].append({"stage_id": "P4", "status": "skipped"})

pipeline_summary["finished_at"] = datetime.now(timezone.utc).isoformat()
summary_path = RUN_DIR / "pipeline_summary.json"
summary_path.write_text(json.dumps(pipeline_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(pipeline_summary, indent=2))
print(f"Pipeline summary written to {summary_path}")
